This script needs to evaluate the LDC_LPI datasets and determine the primary mgrs tile for the observation so that we can run k2 harmonics on a buffer of those pixels. We will subset so that we have only spoked plots, buffered by the length of those spokes.

determine how many lines and what spatial support and what source a line is:

In [2]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

LPI_PATH = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\LDC_LPI_georeferenced_2018_present.csv"
)

OUT_DIR = LPI_PATH.parent / "support_inventory"
OUT_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 1_000_000

assert LPI_PATH.exists(), f"File not found:\n{LPI_PATH}"

size_gb = LPI_PATH.stat().st_size / 1024**3

print(f"Input: {LPI_PATH}")
print(f"Size:  {size_gb:,.2f} GiB")
print(f"Output: {OUT_DIR}")

Input: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_georeferenced_2018_present.csv
Size:  5.56 GiB
Output: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\support_inventory


In [3]:
# =============================================================================
# SCHEMA
# =============================================================================

header = pd.read_csv(
    LPI_PATH,
    nrows=0
)

columns = header.columns.tolist()

print(f"Number of columns: {len(columns)}\n")

for i, col in enumerate(columns):
    print(f"{i:>3}: {col}")

Number of columns: 33

  0: rid
  1: PrimaryKey
  2: DBKey.x
  3: ProjectKey.x
  4: LineKey
  5: RecKey
  6: layer
  7: code
  8: chckbox
  9: ShrubShape
 10: FormType
 11: FormDate
 12: Direction
 13: Measure
 14: LineLengthAmount
 15: SpacingIntervalAmount
 16: SpacingType
 17: ShowCheckbox
 18: CheckboxLabel
 19: PointLoc
 20: PointNbr
 21: source.x
 22: DateLoadedInDb
 23: DateVisited
 24: DateVisited_parsed
 25: SampleYear
 26: Latitude_NAD83
 27: Longitude_NAD83
 28: PlotID
 29: ProjectKey.y
 30: source.y
 31: EcologicalSiteID
 32: DBKey.y


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

LPI_PATH = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\LDC_LPI_georeferenced_2018_present.csv"
)

OUT_DIR = LPI_PATH.parent / "support_inventory"
OUT_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 1_000_000

assert LPI_PATH.exists()

print(f"{LPI_PATH.stat().st_size / 1024**3:.2f} GiB")

5.56 GiB


In [6]:
USECOLS = [
    "PrimaryKey",
    "LineKey",
    "Direction",
    "Measure",
    "LineLengthAmount",
    "SpacingIntervalAmount",
    "SpacingType",
    "PointLoc",
    "PointNbr",
    "SampleYear",
    "Latitude_NAD83",
    "Longitude_NAD83",
    "EcologicalSiteID",
]

In [7]:
sample = pd.read_csv(
    LPI_PATH,
    usecols=USECOLS,
    nrows=100_000,
    low_memory=False
)

for col in [
    "Direction",
    "Measure",
    "LineLengthAmount",
    "SpacingIntervalAmount",
    "SpacingType",
    "PointLoc",
]:
    print("\n", "=" * 80)
    print(col)
    print(sample[col].dropna().value_counts().head(30))


Direction
Direction
0      38278
240    30815
120    29532
180      238
110      224
140      150
330      128
135      120
320      119
170      108
300      107
50        93
250       88
Name: count, dtype: int64

Measure
Measure
1    100000
Name: count, dtype: int64

LineLengthAmount
LineLengthAmount
50    99940
20       60
Name: count, dtype: int64

SpacingIntervalAmount
SpacingIntervalAmount
1    100000
Name: count, dtype: int64

SpacingType
SpacingType
m    100000
Name: count, dtype: int64

PointLoc
PointLoc
23    2069
40    2055
20    2053
32    2052
8     2043
10    2038
26    2035
30    2035
25    2033
13    2031
15    2030
29    2030
35    2026
9     2025
37    2021
21    2015
18    2015
22    2015
14    2014
33    2014
24    2005
4     2003
16    2001
34    2001
36    2001
5     2000
47    1998
3     1997
1     1994
43    1992
Name: count, dtype: int64


We need to evaluate the sources of these lines, since we know that AIM plots are 25m transects with 0.5m hit reads, we dont want to assume that every hit is a meter. 

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

LPI_PATH = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\LDC_LPI_georeferenced_2018_present.csv"
)

OUT_DIR = LPI_PATH.parent / "protocol_inventory"
OUT_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 1_000_000

USECOLS = [
    "PrimaryKey",
    "DBKey.x",
    "ProjectKey.x",
    "LineKey",

    "Direction",
    "Measure",
    "LineLengthAmount",
    "SpacingIntervalAmount",
    "SpacingType",
    "PointLoc",
    "PointNbr",

    "source.x",
    "source.y",

    "SampleYear",
    "Latitude_NAD83",
    "Longitude_NAD83",
    "PlotID",

    "EcologicalSiteID",
]

In [9]:
source_x_counts = {}
source_y_counts = {}

reader = pd.read_csv(
    LPI_PATH,
    usecols=["source.x", "source.y"],
    chunksize=CHUNK_SIZE,
    low_memory=False
)

rows = 0

for i, chunk in enumerate(reader, start=1):

    rows += len(chunk)

    for col, store in [
        ("source.x", source_x_counts),
        ("source.y", source_y_counts),
    ]:

        counts = (
            chunk[col]
            .dropna()
            .astype(str)
            .str.strip()
            .value_counts()
        )

        for value, n in counts.items():
            store[value] = store.get(value, 0) + n

    print(
        f"\rChunk {i:>3} | rows scanned: {rows:>12,}",
        end=""
    )

print()

Chunk  17 | rows scanned:   16,150,114


In [10]:
source_x = (
    pd.Series(source_x_counts, name="n_records")
    .sort_values(ascending=False)
    .rename_axis("source_x")
    .reset_index()
)

source_y = (
    pd.Series(source_y_counts, name="n_records")
    .sort_values(ascending=False)
    .rename_axis("source_y")
    .reset_index()
)

print("source.x")
display(source_x)

print("source.y")
display(source_y)

source.x


,source_x,n_records
0,TERRADAT,10217735
1,NRI,2676620
2,LMF,2168574
3,LandPKS,458595
4,AIM,317868
5,DIMA,310722


source.y


,source_y,n_records
0,TERRADAT,10217735
1,NRI,2676620
2,LMF,2168574
3,LandPKS,458595
4,DIMA,334045
5,AIM,294545


In [11]:
project_chunks = []

reader = pd.read_csv(
    LPI_PATH,
    usecols=[
        "PrimaryKey",
        "ProjectKey.x",
        "DBKey.x",
        "source.x",
        "source.y",
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for i, chunk in enumerate(reader, start=1):

    chunk = (
        chunk[
            [
                "PrimaryKey",
                "ProjectKey.x",
                "DBKey.x",
                "source.x",
                "source.y",
            ]
        ]
        .drop_duplicates()
    )

    project_chunks.append(chunk)

    print(f"\rChunk {i:>3}", end="")

print()

Chunk  17


In [12]:
project_inventory = (
    pd.concat(project_chunks, ignore_index=True)
    .drop_duplicates()
)

project_summary = (
    project_inventory
    .groupby(
        [
            "source.x",
            "source.y",
            "ProjectKey.x",
            "DBKey.x",
        ],
        dropna=False
    )
    .agg(
        n_plots=("PrimaryKey", "nunique")
    )
    .reset_index()
    .sort_values("n_plots", ascending=False)
)

display(project_summary.head(100))

,source.x,source.y,ProjectKey.x,DBKey.x,n_plots
125,TERRADAT,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,30294
124,NRI,NRI,NRI,NRI2024,9818
122,LMF,LMF,BLM_AIM,BLM_Natl_AIM_LMF_Public.gdb,8682
123,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,2190
2,AIM,AIM,NDOW_BIFO,BIFO2015-2020AllSites_DIMA5.5.asof20210225,223
...,...,...,...,...,...
92,AIM,DIMA,NWERN_Pullman,REPORT25Sept18PullmanDIMA5.5aasof2020-06-26,1
91,AIM,DIMA,NWERN_Mandan,REPORT15May18MandanDIMA5.5aasof2020-06-26,1
89,AIM,DIMA,NWERN_Akron,REPORT4Apr19AkronDIMA5.5aasof2020-06-26,1
87,AIM,DIMA,NWERN_Akron,REPORT12Mar19AkronDIMA5.5aasof2020-06-26,1


In [13]:
line_chunks = []

reader = pd.read_csv(
    LPI_PATH,
    usecols=USECOLS,
    chunksize=CHUNK_SIZE,
    low_memory=False
)

rows = 0

for i, chunk in enumerate(reader, start=1):

    rows += len(chunk)

    chunk["PrimaryKey"] = (
        chunk["PrimaryKey"]
        .astype("string")
        .str.strip()
    )

    chunk["LineKey"] = (
        chunk["LineKey"]
        .astype("string")
        .str.strip()
    )

    chunk = chunk.dropna(
        subset=[
            "PrimaryKey",
            "LineKey",
        ]
    )

    for col in [
        "Direction",
        "LineLengthAmount",
        "SpacingIntervalAmount",
        "PointLoc",
        "PointNbr",
        "SampleYear",
        "Latitude_NAD83",
        "Longitude_NAD83",
    ]:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

    g = (
        chunk
        .groupby(
            [
                "PrimaryKey",
                "LineKey",
            ],
            as_index=False
        )
        .agg(
            DBKey=("DBKey.x", "first"),
            ProjectKey=("ProjectKey.x", "first"),

            source_x=("source.x", "first"),
            source_y=("source.y", "first"),

            direction=("Direction", "first"),

            measure=("Measure", "first"),

            line_length_amount=(
                "LineLengthAmount",
                "first"
            ),

            spacing_interval_amount=(
                "SpacingIntervalAmount",
                "first"
            ),

            spacing_type=(
                "SpacingType",
                "first"
            ),

            pointloc_min=("PointLoc", "min"),
            pointloc_max=("PointLoc", "max"),
            n_pointloc=("PointLoc", "nunique"),

            pointnbr_min=("PointNbr", "min"),
            pointnbr_max=("PointNbr", "max"),
            n_pointnbr=("PointNbr", "nunique"),

            sample_year=("SampleYear", "first"),

            latitude=("Latitude_NAD83", "first"),
            longitude=("Longitude_NAD83", "first"),

            PlotID=("PlotID", "first"),

            EcologicalSiteID=(
                "EcologicalSiteID",
                "first"
            ),
        )
    )

    line_chunks.append(g)

    print(
        f"\rChunk {i:>3} | "
        f"rows scanned: {rows:>12,}",
        end=""
    )

print()

Chunk  17 | rows scanned:   16,150,114


In [14]:
line_inventory_raw = pd.concat(
    line_chunks,
    ignore_index=True
)

line_inventory = (
    line_inventory_raw
    .groupby(
        [
            "PrimaryKey",
            "LineKey",
        ],
        as_index=False
    )
    .agg(
        DBKey=("DBKey", "first"),
        ProjectKey=("ProjectKey", "first"),

        source_x=("source_x", "first"),
        source_y=("source_y", "first"),

        direction=("direction", "first"),
        measure=("measure", "first"),

        line_length_amount=(
            "line_length_amount",
            "first"
        ),

        spacing_interval_amount=(
            "spacing_interval_amount",
            "first"
        ),

        spacing_type=("spacing_type", "first"),

        pointloc_min=("pointloc_min", "min"),
        pointloc_max=("pointloc_max", "max"),
        n_pointloc=("n_pointloc", "max"),

        pointnbr_min=("pointnbr_min", "min"),
        pointnbr_max=("pointnbr_max", "max"),
        n_pointnbr=("n_pointnbr", "max"),

        sample_year=("sample_year", "first"),

        latitude=("latitude", "first"),
        longitude=("longitude", "first"),

        PlotID=("PlotID", "first"),

        EcologicalSiteID=(
            "EcologicalSiteID",
            "first"
        ),
    )
)

print(f"Unique lines: {len(line_inventory):,}")
print(
    f"Unique plot-visits: "
    f"{line_inventory['PrimaryKey'].nunique():,}"
)

line_inventory.to_csv(
    OUT_DIR / "LDC_LPI_line_inventory.csv",
    index=False
)

Unique lines: 172,679
Unique plot-visits: 51,812


In [15]:
protocol_by_source = (
    line_inventory
    .groupby(
        [
            "source_x",
            "line_length_amount",
            "measure",
            "spacing_interval_amount",
            "spacing_type",
        ],
        dropna=False
    )
    .agg(
        n_lines=("LineKey", "size"),
        n_plots=("PrimaryKey", "nunique"),
        n_projects=("ProjectKey", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["source_x", "n_lines"],
        ascending=[True, False]
    )
)

display(protocol_by_source)

,source_x,line_length_amount,measure,spacing_interval_amount,spacing_type,n_lines,n_plots,n_projects
1,AIM,50,1.0,1.0000,m,919,309,3
4,AIM,100,1.0,25.0000,cm,282,96,12
3,AIM,100,1.0,1.0000,m,17,15,2
2,AIM,100,1.0,0.2500,m,6,2,2
5,AIM,100,1.0,50.0000,cm,3,1,1
0,AIM,20,1.0,1.0000,m,1,1,1
11,DIMA,50,1.0,25.0000,cm,336,112,1
8,DIMA,30,NaN,50.0000,cm,169,58,1
7,DIMA,25,1.0,50.0000,cm,168,168,2
14,DIMA,100,1.0,25.0000,cm,121,44,10


In [16]:
protocol_by_source.to_csv(
    OUT_DIR / "LDC_LPI_protocol_by_source.csv",
    index=False
)

In [17]:
point_protocols = (
    line_inventory
    .groupby(
        [
            "source_x",
            "line_length_amount",
            "spacing_interval_amount",
            "spacing_type",
            "pointloc_min",
            "pointloc_max",
            "n_pointloc",
        ],
        dropna=False
    )
    .agg(
        n_lines=("LineKey", "size"),
        n_plots=("PrimaryKey", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["source_x", "n_lines"],
        ascending=[True, False]
    )
)

display(point_protocols.head(200))

,source_x,line_length_amount,spacing_interval_amount,spacing_type,pointloc_min,pointloc_max,n_pointloc,n_lines,n_plots
1,AIM,50,1.0,m,1.0,50.0,50,919,309
7,AIM,100,25.0,cm,0.0,100.0,100,137,69
8,AIM,100,25.0,cm,0.0,100.0,101,121,70
5,AIM,100,1.0,m,1.0,100.0,100,14,14
12,AIM,100,25.0,cm,0.0,100.0,396,9,6
...,...,...,...,...,...,...,...,...,...
76,TERRADAT,25,25.0,cm,25.0,2500.0,66,1,1
77,TERRADAT,25,25.0,cm,25.0,2500.0,67,1,1
78,TERRADAT,25,25.0,cm,25.0,2500.0,69,1,1
81,TERRADAT,25,25.0,cm,25.0,2500.0,72,1,1


In [18]:
def sorted_directions(x):

    vals = (
        pd.to_numeric(x, errors="coerce")
        .dropna()
        .astype(float)
        .mod(360)
        .sort_values()
        .tolist()
    )

    return tuple(vals)


plot_geometry = (
    line_inventory
    .groupby("PrimaryKey")
    .agg(
        source_x=("source_x", "first"),
        source_y=("source_y", "first"),

        ProjectKey=("ProjectKey", "first"),

        n_lines=("LineKey", "nunique"),

        directions=(
            "direction",
            sorted_directions
        ),

        min_line_length=(
            "line_length_amount",
            "min"
        ),

        max_line_length=(
            "line_length_amount",
            "max"
        ),

        min_spacing=(
            "spacing_interval_amount",
            "min"
        ),

        max_spacing=(
            "spacing_interval_amount",
            "max"
        ),

        sample_year=("sample_year", "first"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    .reset_index()
)

In [19]:
geometry_by_source = (
    plot_geometry
    .groupby(
        [
            "source_x",
            "n_lines",
            "directions",
            "min_line_length",
            "max_line_length",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_plots")
    .sort_values(
        ["source_x", "n_plots"],
        ascending=[True, False]
    )
)

display(geometry_by_source.head(200))

,source_x,n_lines,directions,min_line_length,max_line_length,n_plots
29,AIM,3,"(0.0, 120.0, 240.0)",50,50,268
26,AIM,3,"(0.0, 60.0, 120.0)",100,100,77
19,AIM,3,"(0.0, 0.0, 0.0)",100,100,20
18,AIM,3,"(0.0, 0.0, 0.0)",50,50,18
23,AIM,3,"(0.0, 0.0, 240.0)",50,50,4
...,...,...,...,...,...,...
143,TERRADAT,2,"(20.0, 280.0)",25,25,1
144,TERRADAT,2,"(21.0, 180.0)",25,25,1
145,TERRADAT,2,"(21.0, 346.0)",25,25,1
146,TERRADAT,2,"(22.0, 290.0)",25,25,1


In [20]:
source_summary = (
    plot_geometry
    .groupby("source_x", dropna=False)
    .agg(
        n_plots=("PrimaryKey", "nunique"),
        n_projects=("ProjectKey", "nunique"),

        median_lines=("n_lines", "median"),
        min_lines=("n_lines", "min"),
        max_lines=("n_lines", "max"),

        median_nominal_length=(
            "max_line_length",
            "median"
        ),
    )
    .reset_index()
    .sort_values(
        "n_plots",
        ascending=False
    )
)

display(source_summary)

,source_x,n_plots,n_projects,median_lines,min_lines,max_lines,median_nominal_length
5,TERRADAT,30294,1,3.0,1,5,25.0
4,NRI,9818,1,2.0,1,2,45.0
2,LMF,8682,1,2.0,1,2,45.0
3,LandPKS,2190,1,20.0,20,20,1.0
0,AIM,422,15,3.0,1,3,50.0
1,DIMA,406,19,3.0,1,3,30.0


In [21]:
# =============================================================================
# ANALYSIS LINE INVENTORY
# Exclude LandPKS only; retain all other protocols for diagnosis.
# =============================================================================

analysis_lines = (
    line_inventory[
        line_inventory["source_x"].ne("LandPKS")
    ]
    .copy()
)

print(f"All unique lines:       {len(line_inventory):,}")
print(f"After LandPKS removal:  {len(analysis_lines):,}")

print(
    "\nPlots by source:"
)

display(
    analysis_lines
    .groupby("source_x")["PrimaryKey"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_plots")
    .reset_index()
)

All unique lines:       172,679
After LandPKS removal:  128,879

Plots by source:


,source_x,n_plots
0,TERRADAT,30294
1,NRI,9818
2,LMF,8682
3,AIM,423
4,DIMA,410


In [22]:
# =============================================================================
# PLOT-LEVEL PROTOCOL TABLE
# =============================================================================

def sorted_tuple(x):
    vals = (
        pd.to_numeric(x, errors="coerce")
        .dropna()
        .astype(float)
        .sort_values()
        .tolist()
    )
    return tuple(vals)


def rounded_tuple(x, decimals=4):
    vals = (
        pd.to_numeric(x, errors="coerce")
        .dropna()
        .astype(float)
        .round(decimals)
        .sort_values()
        .tolist()
    )
    return tuple(vals)


plot_protocol = (
    analysis_lines
    .groupby("PrimaryKey", as_index=False)
    .agg(
        source_x=("source_x", "first"),
        source_y=("source_y", "first"),

        ProjectKey=("ProjectKey", "first"),
        DBKey=("DBKey", "first"),
        PlotID=("PlotID", "first"),

        sample_year=("sample_year", "first"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),

        n_lines=("LineKey", "nunique"),

        directions=("direction", sorted_tuple),

        line_lengths=(
            "line_length_amount",
            rounded_tuple
        ),

        spacing_intervals=(
            "spacing_interval_amount",
            rounded_tuple
        ),

        spacing_types=(
            "spacing_type",
            lambda x: tuple(
                sorted(
                    x.dropna()
                     .astype(str)
                     .unique()
                )
            )
        ),

        pointloc_min=("pointloc_min", "min"),
        pointloc_max=("pointloc_max", "max"),

        min_n_pointloc=("n_pointloc", "min"),
        max_n_pointloc=("n_pointloc", "max"),

        ecological_site_id=(
            "EcologicalSiteID",
            "first"
        ),
    )
)

print(f"Plot-visits retained: {len(plot_protocol):,}")

display(plot_protocol.head())

Plot-visits retained: 49,622


,PrimaryKey,source_x,source_y,ProjectKey,DBKey,PlotID,sample_year,latitude,longitude,n_lines,directions,line_lengths,spacing_intervals,spacing_types,pointloc_min,pointloc_max,min_n_pointloc,max_n_pointloc,ecological_site_id
0,1000000032018-09-01,TERRADAT,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,Bottom-009,2018,31.880350,-106.979390,3,"(72.0, 192.0, 312.0)","(25.0, 25.0, 25.0)","(25.0, 25.0, 25.0)","(cm,)",0.25,25.0,100,100,R042XB023NM
1,1000000302018-09-01,TERRADAT,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,GSand-150,2018,32.214021,-106.674129,3,"(20.0, 140.0, 260.0)","(25.0, 25.0, 25.0)","(25.0, 25.0, 25.0)","(cm,)",0.25,25.0,100,100,R042XB024NM
2,1000000692018-09-01,TERRADAT,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,Loam-341,2018,31.805752,-107.439835,3,"(36.0, 156.0, 276.0)","(25.0, 25.0, 25.0)","(25.0, 25.0, 25.0)","(cm,)",0.25,25.0,100,100,R042XB012NM
3,1000000772018-09-01,TERRADAT,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,Mal-389,2018,32.021330,-106.930920,3,"(78.0, 198.0, 318.0)","(25.0, 25.0, 25.0)","(25.0, 25.0, 25.0)","(cm,)",0.25,25.0,73,84,R042XB015NM
4,1000000962018-09-01,TERRADAT,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,Sandy-488,2018,32.116290,-107.238020,3,"(60.0, 180.0, 300.0)","(25.0, 25.0, 25.0)","(25.0, 25.0, 25.0)","(cm,)",0.25,25.0,100,100,R042XB015NM


In [23]:
# =============================================================================
# LEGACY AIM DIAGNOSTICS
# =============================================================================

aim = plot_protocol[
    plot_protocol["source_x"].eq("AIM")
].copy()

print(f"AIM plots: {len(aim):,}")

AIM plots: 422


In [24]:
aim_protocols = (
    aim
    .groupby(
        [
            "ProjectKey",
            "n_lines",
            "directions",
            "line_lengths",
            "spacing_intervals",
            "spacing_types",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_plots")
    .sort_values(
        ["n_plots"],
        ascending=False
    )
)

display(aim_protocols.head(100))

,ProjectKey,n_lines,directions,line_lengths,spacing_intervals,spacing_types,n_plots
30,NDOW_BIFO,3,"(0.0, 120.0, 240.0)","(50.0, 50.0, 50.0)","(1.0, 1.0, 1.0)","(m,)",198
22,NDOW,3,"(0.0, 120.0, 240.0)","(50.0, 50.0, 50.0)","(1.0, 1.0, 1.0)","(m,)",70
42,NWERN_HAFB,3,"(0.0, 60.0, 120.0)","(100.0, 100.0, 100.0)","(25.0, 25.0, 25.0)","(cm,)",17
43,NWERN_JER,3,"(0.0, 60.0, 120.0)","(100.0, 100.0, 100.0)","(25.0, 25.0, 25.0)","(cm,)",17
25,NDOW_BIFO,3,"(0.0, 0.0, 0.0)","(50.0, 50.0, 50.0)","(1.0, 1.0, 1.0)","(m,)",13
49,NWERN_Pullman,3,"(0.0, 60.0, 120.0)","(100.0, 100.0, 100.0)","(25.0, 25.0, 25.0)","(cm,)",7
44,NWERN_Lordsburg,3,"(0.0, 0.0, 0.0)","(100.0, 100.0, 100.0)","(25.0, 25.0, 25.0)","(cm,)",6
53,NWERN_SLV,3,"(0.0, 60.0, 120.0)","(100.0, 100.0, 100.0)","(25.0, 25.0, 25.0)","(cm,)",5
37,NWERN_Akron,3,"(0.0, 60.0, 120.0)","(100.0, 100.0, 100.0)","(25.0, 25.0, 25.0)","(cm,)",5
46,NWERN_Mandan,3,"(0.0, 60.0, 120.0)","(100.0, 100.0, 100.0)","(25.0, 25.0, 25.0)","(cm,)",5


In [25]:
# =============================================================================
# ASSIGN VERIFIED PROTOCOL CLASSES
# =============================================================================

plot_protocol["protocol_class"] = "UNRESOLVED"
plot_protocol["support_radius_m"] = np.nan
plot_protocol["include_for_gee"] = False

In [26]:
# Standard current AIM: 3 radial 25 m transects

is_terradat_aim25 = (
    plot_protocol["source_x"].eq("TERRADAT")
    &
    plot_protocol["n_lines"].eq(3)
    &
    plot_protocol["line_lengths"].apply(
        lambda x: len(x) == 3 and all(v == 25 for v in x)
    )
)

plot_protocol.loc[
    is_terradat_aim25,
    ["protocol_class", "support_radius_m", "include_for_gee"]
] = [
    "TERRADAT_AIM_25M",
    25.0,
    True
]

In [27]:
# NRI: two intersecting 150-ft transects centered on plot

is_nri = (
    plot_protocol["source_x"].eq("NRI")
    &
    plot_protocol["n_lines"].eq(2)
)

plot_protocol.loc[
    is_nri,
    ["protocol_class", "support_radius_m", "include_for_gee"]
] = [
    "NRI_CROSSED_150FT",
    22.86,
    True
]

In [28]:
# LMF uses NRI geometry per BLM documentation

is_lmf = (
    plot_protocol["source_x"].eq("LMF")
    &
    plot_protocol["n_lines"].eq(2)
)

plot_protocol.loc[
    is_lmf,
    ["protocol_class", "support_radius_m", "include_for_gee"]
] = [
    "LMF_CROSSED_150FT",
    22.86,
    True
]

In [30]:
def approx_three_spoke(directions, tolerance=15):
    if len(directions) != 3:
        return False

    d = np.sort(np.asarray(directions) % 360)
    gaps = np.diff(np.r_[d, d[0] + 360])

    return np.all(
        np.abs(gaps - 120) <= tolerance
    )

In [31]:
is_aim_legacy50 = (
    plot_protocol["source_x"].eq("AIM")
    &
    plot_protocol["n_lines"].eq(3)
    &
    plot_protocol["line_lengths"].apply(
        lambda x: len(x) == 3 and all(v == 50 for v in x)
    )
    &
    plot_protocol["directions"].apply(approx_three_spoke)
)

plot_protocol.loc[
    is_aim_legacy50,
    ["protocol_class", "support_radius_m", "include_for_gee"]
] = [
    "AIM_LEGACY_50M",
    50.0,
    True
]

In [32]:
support_summary = (
    plot_protocol
    .groupby(
        ["protocol_class", "include_for_gee"],
        dropna=False
    )
    .size()
    .reset_index(name="n_plots")
    .sort_values("n_plots", ascending=False)
)

display(support_summary)

,protocol_class,include_for_gee,n_plots
3,TERRADAT_AIM_25M,True,29427
2,NRI_CROSSED_150FT,True,9812
1,LMF_CROSSED_150FT,True,8681
4,UNRESOLVED,False,1430
0,AIM_LEGACY_50M,True,272


In [33]:
gee_sites = (
    plot_protocol[
        plot_protocol["include_for_gee"]
    ]
    .copy()
)

print(f"Resolved GEE-ready plots: {len(gee_sites):,}")

Resolved GEE-ready plots: 48,192


In [34]:
gee_sites["support_area_m2"] = (
    np.pi * gee_sites["support_radius_m"]**2
)

In [36]:
import mgrs

m = mgrs.MGRS()

def get_mgrs_tile(lat, lon):
    tile = m.toMGRS(
        float(lat),
        float(lon),
        MGRSPrecision=0
    )

    if isinstance(tile, bytes):
        tile = tile.decode("utf-8")

    return tile

In [37]:
gee_sites["primary_mgrs_tile"] = [
    get_mgrs_tile(lat, lon)
    for lat, lon in zip(
        gee_sites["latitude"],
        gee_sites["longitude"]
    )
]

In [38]:
gee_sites["job_id"] = (
    gee_sites["sample_year"]
    .astype(int)
    .astype(str)
    + "_"
    + gee_sites["primary_mgrs_tile"]
)

In [39]:
job_summary = (
    gee_sites
    .groupby(
        [
            "sample_year",
            "primary_mgrs_tile",
            "protocol_class"
        ],
        as_index=False
    )
    .agg(
        n_plots=("PrimaryKey", "size"),
        min_support_radius_m=("support_radius_m", "min"),
        max_support_radius_m=("support_radius_m", "max"),
    )
    .sort_values(
        [
            "sample_year",
            "primary_mgrs_tile",
            "protocol_class"
        ]
    )
)

display(job_summary)

,sample_year,primary_mgrs_tile,protocol_class,n_plots,min_support_radius_m,max_support_radius_m
0,2018,06VUN,TERRADAT_AIM_25M,14,25.00,25.00
1,2018,06WUA,TERRADAT_AIM_25M,4,25.00,25.00
2,2018,06WVT,TERRADAT_AIM_25M,2,25.00,25.00
3,2018,06WWT,TERRADAT_AIM_25M,1,25.00,25.00
4,2018,06WXT,TERRADAT_AIM_25M,3,25.00,25.00
...,...,...,...,...,...,...
6350,2024,18TYN,NRI_CROSSED_150FT,1,22.86,22.86
6351,2024,18TYP,NRI_CROSSED_150FT,1,22.86,22.86
6352,2024,19TBG,NRI_CROSSED_150FT,2,22.86,22.86
6353,2024,19TBH,NRI_CROSSED_150FT,1,22.86,22.86


In [40]:
year_summary = (
    gee_sites
    .groupby("sample_year", as_index=False)
    .agg(
        n_plots=("PrimaryKey", "size"),
        n_mgrs_tiles=("primary_mgrs_tile", "nunique"),
        n_protocols=("protocol_class", "nunique"),
    )
)

display(year_summary)

,sample_year,n_plots,n_mgrs_tiles,n_protocols
0,2018,6655,654,4
1,2019,8545,665,4
2,2020,7098,596,4
3,2021,8233,640,4
4,2022,8487,632,3
5,2023,7974,639,3
6,2024,1200,474,2


In [41]:
year_protocol_summary = (
    gee_sites
    .groupby(
        ["sample_year", "protocol_class"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "n_plots"})
)

display(year_protocol_summary)

,sample_year,protocol_class,n_plots
0,2018,AIM_LEGACY_50M,53
1,2018,LMF_CROSSED_150FT,1546
2,2018,NRI_CROSSED_150FT,1695
3,2018,TERRADAT_AIM_25M,3361
4,2019,AIM_LEGACY_50M,16
5,2019,LMF_CROSSED_150FT,1561
6,2019,NRI_CROSSED_150FT,1614
7,2019,TERRADAT_AIM_25M,5354
8,2020,AIM_LEGACY_50M,131
9,2020,LMF_CROSSED_150FT,1316


In [42]:
# =============================================================================
# 1. FINAL QA BEFORE FREEZING THE GEE-READY DATASET
# =============================================================================

required_cols = [
    "PrimaryKey",
    "sample_year",
    "latitude",
    "longitude",
    "source_x",
    "ProjectKey",
    "protocol_class",
    "n_lines",
    "directions",
    "support_radius_m",
    "primary_mgrs_tile",
]

missing = [
    c for c in required_cols
    if c not in gee_sites.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

assert gee_sites["PrimaryKey"].is_unique
assert gee_sites["support_radius_m"].notna().all()
assert gee_sites["primary_mgrs_tile"].notna().all()

assert gee_sites["sample_year"].between(
    2018, 2024
).all()

print(f"GEE-ready sites: {len(gee_sites):,}")
print(
    f"Unique PrimaryKeys: "
    f"{gee_sites['PrimaryKey'].nunique():,}"
)

GEE-ready sites: 48,192
Unique PrimaryKeys: 48,192


In [43]:
# =============================================================================
# 2. EXPLICIT PHYSICAL-SUPPORT METADATA
# =============================================================================

gee_sites["support_area_m2"] = (
    np.pi *
    gee_sites["support_radius_m"]**2
)

gee_sites["expected_10m_pixel_equiv"] = (
    gee_sites["support_area_m2"] / 100.0
)

gee_sites["support_geometry"] = (
    gee_sites["protocol_class"]
    .map({
        "TERRADAT_AIM_25M":
            "3_radial_spokes",

        "AIM_LEGACY_50M":
            "3_radial_spokes",

        "NRI_CROSSED_150FT":
            "2_crossed_transects",

        "LMF_CROSSED_150FT":
            "2_crossed_transects",
    })
)

In [44]:
# Approximate protocol-specific design metadata.
# These describe the verified physical protocol, not raw database fields.

gee_sites["transect_length_m"] = np.nan
gee_sites["nominal_point_spacing_m"] = np.nan

gee_sites.loc[
    gee_sites["protocol_class"].eq(
        "TERRADAT_AIM_25M"
    ),
    ["transect_length_m",
     "nominal_point_spacing_m"]
] = [25.0, 0.5]


gee_sites.loc[
    gee_sites["protocol_class"].eq(
        "AIM_LEGACY_50M"
    ),
    ["transect_length_m",
     "nominal_point_spacing_m"]
] = [50.0, 1.0]


gee_sites.loc[
    gee_sites["protocol_class"].isin([
        "NRI_CROSSED_150FT",
        "LMF_CROSSED_150FT",
    ]),
    ["transect_length_m",
     "nominal_point_spacing_m"]
] = [45.72, 0.9144]

In [45]:
# =============================================================================
# 3. RETAIN RAW PROTOCOL METADATA
# =============================================================================

raw_protocol = (
    plot_protocol[
        [
            "PrimaryKey",
            "line_lengths",
            "spacing_intervals",
            "spacing_types",
        ]
    ]
    .copy()
)

gee_sites = gee_sites.merge(
    raw_protocol,
    on="PrimaryKey",
    how="left",
    validate="one_to_one"
)

In [46]:
# =============================================================================
# 4. STABLE PROCESSING IDENTIFIERS
# =============================================================================

gee_sites["sample_year"] = (
    gee_sites["sample_year"]
    .astype(int)
)

gee_sites["site_year_id"] = (
    gee_sites["PrimaryKey"].astype(str)
    + "_"
    + gee_sites["sample_year"].astype(str)
)

gee_sites["year_mgrs_id"] = (
    gee_sites["sample_year"].astype(str)
    + "_"
    + gee_sites["primary_mgrs_tile"]
)

In [48]:
print(gee_sites.columns.tolist())

['PrimaryKey', 'source_x', 'source_y', 'ProjectKey', 'DBKey', 'PlotID', 'sample_year', 'latitude', 'longitude', 'n_lines', 'directions', 'line_lengths_x', 'spacing_intervals_x', 'spacing_types_x', 'pointloc_min', 'pointloc_max', 'min_n_pointloc', 'max_n_pointloc', 'ecological_site_id', 'protocol_class', 'support_radius_m', 'include_for_gee', 'support_area_m2', 'primary_mgrs_tile', 'job_id', 'expected_10m_pixel_equiv', 'support_geometry', 'transect_length_m', 'nominal_point_spacing_m', 'line_lengths_y', 'spacing_intervals_y', 'spacing_types_y', 'site_year_id', 'year_mgrs_id']


In [49]:
# =============================================================================
# CHECK WHETHER _x AND _y COPIES MATCH
# =============================================================================

pairs = [
    ("line_lengths_x", "line_lengths_y"),
    ("spacing_intervals_x", "spacing_intervals_y"),
    ("spacing_types_x", "spacing_types_y"),
]

for a, b in pairs:
    same = gee_sites[a].astype(str).eq(
        gee_sites[b].astype(str)
    )

    print(
        f"{a} vs {b}: "
        f"{same.sum():,} / {len(same):,} identical"
    )

    if not same.all():
        print(
            gee_sites.loc[
                ~same,
                ["PrimaryKey", a, b]
            ].head(20)
        )

line_lengths_x vs line_lengths_y: 48,192 / 48,192 identical
spacing_intervals_x vs spacing_intervals_y: 48,192 / 48,192 identical
spacing_types_x vs spacing_types_y: 48,192 / 48,192 identical


In [50]:
# =============================================================================
# CLEAN DUPLICATED PROTOCOL COLUMNS
# =============================================================================

gee_sites["line_lengths"] = (
    gee_sites["line_lengths_x"]
)

gee_sites["spacing_intervals"] = (
    gee_sites["spacing_intervals_x"]
)

gee_sites["spacing_types"] = (
    gee_sites["spacing_types_x"]
)

gee_sites = gee_sites.drop(
    columns=[
        "line_lengths_x",
        "line_lengths_y",
        "spacing_intervals_x",
        "spacing_intervals_y",
        "spacing_types_x",
        "spacing_types_y",
    ]
)

print(gee_sites.columns.tolist())

['PrimaryKey', 'source_x', 'source_y', 'ProjectKey', 'DBKey', 'PlotID', 'sample_year', 'latitude', 'longitude', 'n_lines', 'directions', 'pointloc_min', 'pointloc_max', 'min_n_pointloc', 'max_n_pointloc', 'ecological_site_id', 'protocol_class', 'support_radius_m', 'include_for_gee', 'support_area_m2', 'primary_mgrs_tile', 'job_id', 'expected_10m_pixel_equiv', 'support_geometry', 'transect_length_m', 'nominal_point_spacing_m', 'site_year_id', 'year_mgrs_id', 'line_lengths', 'spacing_intervals', 'spacing_types']


In [51]:
# =============================================================================
# 5. WRITE MASTER HARMONIC-EXTRACTION MANIFEST
# =============================================================================

MANIFEST_DIR = (
    LPI_PATH.parent /
    "K2_manifests"
)

MANIFEST_DIR.mkdir(
    exist_ok=True
)

master_cols = [
    "PrimaryKey",
    "site_year_id",

    "sample_year",

    "latitude",
    "longitude",

    "source_x",
    "ProjectKey",

    "protocol_class",
    "support_geometry",

    "n_lines",
    "directions",

    "transect_length_m",
    "nominal_point_spacing_m",

    "support_radius_m",
    "support_area_m2",
    "expected_10m_pixel_equiv",

    "primary_mgrs_tile",
    "year_mgrs_id",

    "line_lengths",
    "spacing_intervals",
    "spacing_types",
]

master_manifest = (
    gee_sites[master_cols]
    .sort_values(
        [
            "sample_year",
            "primary_mgrs_tile",
            "PrimaryKey",
        ]
    )
    .reset_index(drop=True)
)

master_path = (
    MANIFEST_DIR /
    "LDC_LPI_K2_master_manifest.csv"
)

master_manifest.to_csv(
    master_path,
    index=False
)

print(master_path)
print(f"{len(master_manifest):,} rows")

C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\LDC_LPI_K2_master_manifest.csv
48,192 rows


we have subset appropriately and retained relevant protocols with similar spatial support. now lets subset by year so GEE can run harmonics on each point with the right sized buffer for that site.

In [52]:
from pathlib import Path
import pandas as pd

MASTER = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\K2_manifests\LDC_LPI_K2_master_manifest.csv"
)

OUT_DIR = MASTER.parent / "annual"
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(MASTER)

print(f"Master rows: {len(df):,}")
print(df["sample_year"].value_counts().sort_index())

Master rows: 48,192
sample_year
2018    6655
2019    8545
2020    7098
2021    8233
2022    8487
2023    7974
2024    1200
Name: count, dtype: int64


In [53]:
for year, g in df.groupby("sample_year"):

    g = (
        g.sort_values(
            ["primary_mgrs_tile", "PrimaryKey"]
        )
        .reset_index(drop=True)
    )

    out = OUT_DIR / f"LDC_LPI_K2_{int(year)}_manifest.csv"

    g.to_csv(out, index=False)

    print(
        f"{int(year)}: "
        f"{len(g):,} plots | "
        f"{g['primary_mgrs_tile'].nunique():,} primary MGRS tiles | "
        f"{out}"
    )

2018: 6,655 plots | 654 primary MGRS tiles | C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2018_manifest.csv
2019: 8,545 plots | 665 primary MGRS tiles | C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2019_manifest.csv
2020: 7,098 plots | 596 primary MGRS tiles | C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2020_manifest.csv
2021: 8,233 plots | 640 primary MGRS tiles | C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2021_manifest.csv
2022: 8,487 plots | 632 primary MGRS tiles | C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2022_manifest.csv
2023: 7,974 plots | 639 primary MGRS tiles | C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2023_manifest.csv
2024: 1,200 plots | 474 primary MGRS tiles | C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2024_manifest.csv

In [54]:
annual_total = 0

for path in sorted(OUT_DIR.glob("LDC_LPI_K2_*_manifest.csv")):

    tmp = pd.read_csv(path)

    annual_total += len(tmp)

    assert tmp["sample_year"].nunique() == 1
    assert tmp["PrimaryKey"].is_unique
    assert tmp["support_radius_m"].notna().all()
    assert tmp["primary_mgrs_tile"].notna().all()

    print(
        path.name,
        len(tmp),
        sorted(tmp["protocol_class"].unique())
    )

print(f"\nAnnual total: {annual_total:,}")

assert annual_total == len(df)

LDC_LPI_K2_2018_manifest.csv 6655 ['AIM_LEGACY_50M', 'LMF_CROSSED_150FT', 'NRI_CROSSED_150FT', 'TERRADAT_AIM_25M']
LDC_LPI_K2_2019_manifest.csv 8545 ['AIM_LEGACY_50M', 'LMF_CROSSED_150FT', 'NRI_CROSSED_150FT', 'TERRADAT_AIM_25M']
LDC_LPI_K2_2020_manifest.csv 7098 ['AIM_LEGACY_50M', 'LMF_CROSSED_150FT', 'NRI_CROSSED_150FT', 'TERRADAT_AIM_25M']
LDC_LPI_K2_2021_manifest.csv 8233 ['AIM_LEGACY_50M', 'LMF_CROSSED_150FT', 'NRI_CROSSED_150FT', 'TERRADAT_AIM_25M']
LDC_LPI_K2_2022_manifest.csv 8487 ['LMF_CROSSED_150FT', 'NRI_CROSSED_150FT', 'TERRADAT_AIM_25M']
LDC_LPI_K2_2023_manifest.csv 7974 ['LMF_CROSSED_150FT', 'NRI_CROSSED_150FT', 'TERRADAT_AIM_25M']
LDC_LPI_K2_2024_manifest.csv 1200 ['NRI_CROSSED_150FT', 'TERRADAT_AIM_25M']

Annual total: 48,192


In [61]:
from pathlib import Path
import subprocess

MANIFEST_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\K2_manifests\annual"
)

# Use an existing bucket you control.
# Change this to the actual bucket name.
GCS_BUCKET = "bop-nca-data-space"

GCS_PREFIX = "LDC_LPI_K2_manifests"

EE_ASSET_ROOT = (
    "projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests"
)

YEARS = range(2018, 2025)

In [64]:
# =============================================================================
# UPLOAD ANNUAL MANIFESTS TO GCS WITH PYTHON
# =============================================================================

from pathlib import Path
from google.cloud import storage

MANIFEST_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\K2_manifests\annual"
)

GCS_BUCKET = "bop-nca-data-space"

GCS_PREFIX = "LDC_LPI_K2_manifests"

YEARS = range(2018, 2025)


# Uses your existing Google authentication.
client = storage.Client()

bucket = client.bucket(GCS_BUCKET)


for year in YEARS:

    local_csv = (
        MANIFEST_DIR /
        f"LDC_LPI_K2_{year}_manifest.csv"
    )

    if not local_csv.exists():
        print(f"{year}: local file missing — skipped")
        continue

    blob_name = (
        f"{GCS_PREFIX}/"
        f"LDC_LPI_K2_{year}_manifest.csv"
    )

    blob = bucket.blob(blob_name)

    print("\n" + "=" * 90)
    print(f"Uploading {year}")
    print(f"Local: {local_csv}")
    print(f"GCS:   gs://{GCS_BUCKET}/{blob_name}")
    print("=" * 90)

    blob.upload_from_filename(
        str(local_csv),
        content_type="text/csv"
    )

    print("Upload complete.")


Uploading 2018
Local: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2018_manifest.csv
GCS:   gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2018_manifest.csv
Upload complete.

Uploading 2019
Local: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2019_manifest.csv
GCS:   gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2019_manifest.csv
Upload complete.

Uploading 2020
Local: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2020_manifest.csv
GCS:   gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2020_manifest.csv
Upload complete.

Uploading 2021
Local: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2021_manifest.csv
GCS:   gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2021_manifest.csv
Upload complete.

Uploading 2022
Local: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\K2_manifests\annual\LDC_LPI_K2_2022_manifest.csv
GCS

In [65]:
# =============================================================================
# VERIFY GCS STAGING FILES
# =============================================================================

blobs = client.list_blobs(
    GCS_BUCKET,
    prefix=GCS_PREFIX + "/"
)

for blob in blobs:
    print(
        f"gs://{GCS_BUCKET}/{blob.name} "
        f"({blob.size:,} bytes)"
    )

gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2018_manifest.csv (1,808,151 bytes)
gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2019_manifest.csv (2,334,995 bytes)
gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2020_manifest.csv (2,014,147 bytes)
gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2021_manifest.csv (2,443,257 bytes)
gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2022_manifest.csv (2,425,673 bytes)
gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2023_manifest.csv (2,293,245 bytes)
gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2024_manifest.csv (338,437 bytes)


In [69]:
import subprocess

EE_ROOT = "projects/bop-nca-data-space/assets"
EE_FOLDER = "projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests"

for path in [EE_ROOT, EE_FOLDER]:

    print("\n", "=" * 80)
    print(path)
    print("=" * 80)

    result = subprocess.run(
        ["earthengine", "asset", "info", path],
        capture_output=True,
        text=True
    )

    print("RETURN CODE:", result.returncode)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)


projects/bop-nca-data-space/assets
RETURN CODE: 0
STDOUT:
{
  "id": "projects/bop-nca-data-space/assets",
  "name": "projects/bop-nca-data-space/assets",
  "quota": {
    "assetCount": "581",
    "maxAssets": "50000",
    "maxSizeBytes": "268435456000",
    "sizeBytes": "232769972371"
  },
  "type": "FOLDER"
}

STDERR:


projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests
RETURN CODE: 0
STDOUT:
{
  "id": "projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests",
  "name": "projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests",
  "type": "FOLDER"
}

STDERR:



In [70]:
# =============================================================================
# INGEST GCS MANIFESTS AS EARTH ENGINE TABLE ASSETS
# =============================================================================

import subprocess

EE_ASSET_ROOT = (
    "projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests"
)

for year in YEARS:

    gcs_uri = (
        f"gs://{GCS_BUCKET}/"
        f"{GCS_PREFIX}/"
        f"LDC_LPI_K2_{year}_manifest.csv"
    )

    asset_id = (
        f"{EE_ASSET_ROOT}/"
        f"LDC_LPI_K2_{year}_manifest"
    )

    print("\n" + "=" * 90)
    print(f"Earth Engine ingestion: {year}")
    print(f"Source: {gcs_uri}")
    print(f"Asset:  {asset_id}")
    print("=" * 90)

    cmd = [
        "earthengine",
        "upload",
        "table",
        f"--asset_id={asset_id}",
        "--x_column=longitude",
        "--y_column=latitude",
        gcs_uri,
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print("ERROR:")
        print(result.stderr)


Earth Engine ingestion: 2018
Source: gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2018_manifest.csv
Asset:  projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests/LDC_LPI_K2_2018_manifest
Started upload task with ID: IS6VREA6TQZALXOGWKGERVUP


Earth Engine ingestion: 2019
Source: gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2019_manifest.csv
Asset:  projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests/LDC_LPI_K2_2019_manifest
Started upload task with ID: QO44ZEXPQZQXI3TFH5XAZAKG


Earth Engine ingestion: 2020
Source: gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2020_manifest.csv
Asset:  projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests/LDC_LPI_K2_2020_manifest
Started upload task with ID: RUBEKA6QJV5O3DVFTXJNAP4G


Earth Engine ingestion: 2021
Source: gs://bop-nca-data-space/LDC_LPI_K2_manifests/LDC_LPI_K2_2021_manifest.csv
Asset:  projects/bop-nca-data-space/assets/LDC_LPI_K2_manifests/LDC_LPI_K2_2021_manifest
Started upload task with ID: MT